In [0]:
%pip install openpyxl

import pandas as pd

BASE = "/Volumes/mvp-base_programação_operacional/bronze/raw_extraction"

In [0]:
spark.sql("USE CATALOG `mvp-base_programação_operacional`")

df_silver = spark.table("silver.agendamentos_unificado").toPandas()

print(df_silver.shape)
df_silver.head()

In [0]:
dim_territorio = (
    df_silver[["territorio_normalizado"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_territorio.insert(0, "sk_territorio", range(1, len(dim_territorio) + 1))
dim_territorio.rename(columns={"territorio_normalizado": "territorio"}, inplace=True)

dim_tipo_atividade = (
    df_silver[["tipo_atividade_normalizado"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_tipo_atividade.insert(0, "sk_tipo_atividade", range(1, len(dim_tipo_atividade) + 1))
dim_tipo_atividade.rename(columns={"tipo_atividade_normalizado": "tipo_atividade"}, inplace=True)

dim_origem_sistema = (
    df_silver[["sistema_origem"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_origem_sistema.insert(0, "sk_origem_sistema", range(1, len(dim_origem_sistema) + 1))

print("dim_territorio:", dim_territorio.shape)
print("dim_tipo_atividade:", dim_tipo_atividade.shape)
print("dim_origem_sistema:", dim_origem_sistema.shape)

In [0]:
df_silver["data_str"] = pd.to_datetime(df_silver["data_agendamento"], errors="coerce").dt.strftime("%Y-%m-%d")
df_silver["data_str"] = df_silver["data_str"].fillna("SEM_DATA")

dim_tempo = (
    df_silver[df_silver["data_str"] != "SEM_DATA"][["data_str"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_tempo.insert(0, "sk_tempo", range(1, len(dim_tempo) + 1))
dim_tempo["data_agendamento"] = pd.to_datetime(dim_tempo["data_str"])
dim_tempo["ano"] = dim_tempo["data_agendamento"].dt.year
dim_tempo["mes"] = dim_tempo["data_agendamento"].dt.month
dim_tempo["dia_semana"] = dim_tempo["data_agendamento"].dt.day_name()

linha_sentinela = pd.DataFrame({
    "sk_tempo": [-1],
    "data_str": ["SEM_DATA"],
    "data_agendamento": [pd.NaT],
    "ano": [None],
    "mes": [None],
    "dia_semana": ["Não aplicável"],
})
dim_tempo = pd.concat([dim_tempo, linha_sentinela], ignore_index=True)

print("dim_tempo:", dim_tempo.shape)
dim_tempo.tail()

In [0]:
mapa_data_para_sk = dict(zip(dim_tempo["data_str"], dim_tempo["sk_tempo"]))

fato_agendamento = df_silver.merge(
    dim_territorio, left_on="territorio_normalizado", right_on="territorio", how="left"
).merge(
     dim_tipo_atividade, left_on="tipo_atividade_normalizado", right_on="tipo_atividade", how="left"
).merge(
    dim_origem_sistema, on="sistema_origem", how="left"
)

fato_agendamento["sk_tempo"] = fato_agendamento["data_str"].map(mapa_data_para_sk)

fato_agendamento.insert(0, "sk_agendamento", range(1, len(fato_agendamento) + 1))

fato_agendamento = fato_agendamento[[
    "sk_agendamento", "id_contrato", "sk_territorio", "sk_tempo",
    "sk_tipo_atividade", "sk_origem_sistema", "documento", "tipo_documento",
    "status_unificado", "valor_recorrente", "flag_replicado_multissistema",
]]

print("Total de linhas na fato:", len(fato_agendamento))
print("Linhas sem território mapeado:", fato_agendamento["sk_territorio"].isna().sum())
print("Linhas com sk_tempo = -1 (sem data):", (fato_agendamento["sk_tempo"] == -1).sum())
fato_agendamento.head(10)

In [0]:
fato_agendamento["documento"] = fato_agendamento["documento"].apply(
    lambda x: str(int(float(x))) if pd.notna(x) else None
)

if -1 not in dim_tipo_atividade["sk_tipo_atividade"].values:
    dim_tipo_atividade = pd.concat([
        dim_tipo_atividade,
        pd.DataFrame({"sk_tipo_atividade": [-1], "tipo_atividade": ["Não informado"]}),
    ], ignore_index=True)

fato_agendamento["sk_tipo_atividade"] = fato_agendamento["sk_tipo_atividade"].fillna(-1).astype(int)

print(dim_tipo_atividade)
print(fato_agendamento["sk_tipo_atividade"].value_counts())

In [0]:
spark.createDataFrame(dim_territorio).write.mode("overwrite").saveAsTable("gold.dim_territorio")
spark.createDataFrame(dim_tipo_atividade).write.mode("overwrite").saveAsTable("gold.dim_tipo_atividade")
spark.createDataFrame(dim_origem_sistema).write.mode("overwrite").saveAsTable("gold.dim_origem_sistema")
spark.createDataFrame(dim_tempo.drop(columns=["data_str"])).write.mode("overwrite").saveAsTable("gold.dim_tempo")
spark.createDataFrame(fato_agendamento).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fato_agendamento")

print("5 tabelas gravadas em gold")

In [0]:
spark.sql("USE CATALOG `mvp-base_programação_operacional`")

comentarios_tabelas = {
    "bronze.sistema_a": "Camada Bronze. Export bruto do Sistema A (CRM padrão), sem alteração de valores. Inclui metadados de ingestão.",
    "bronze.sistema_b": "Camada Bronze. Export bruto do Sistema B (CRM regional 1), mistura PF e PJ. Sem alteração de valores.",
    "bronze.sistema_c": "Camada Bronze. Export bruto do Sistema C (CRM regional 2). Nomes de coluna ajustados para compatibilidade Delta.",
    "bronze.sistema_d": "Camada Bronze. Export bruto do Sistema D (planilha de varejo). Não coleta documento nem valor.",
    "silver.agendamentos_unificado": "Camada Silver. União dos 4 sistemas com território, status, tipo de atividade e PF/PJ padronizados. 435 linhas.",
    "gold.fato_agendamento": "Camada Gold. Tabela fato do esquema estrela. Grão: 1 evento de agendamento por contrato por sistema de origem.",
    "gold.dim_territorio": "Camada Gold. Dimensão de território (dado de referência). 5 territórios válidos.",
    "gold.dim_tipo_atividade": "Camada Gold. Dimensão de tipo de atividade, normalizada para instalação e vistoria.",
    "gold.dim_origem_sistema": "Camada Gold. Dimensão de sistema de origem (A, B, C, D).",
    "gold.dim_tempo": "Camada Gold. Dimensão de tempo com ano, mês e dia da semana. sk_tempo = -1 representa contrato sem data.",
}

comentarios_colunas = {
    "gold.fato_agendamento": {
        "sk_agendamento": "Chave surrogate da fato. Sequencial, sem significado de negócio.",
        "id_contrato": "Identificador do contrato no sistema de origem.",
        "sk_territorio": "FK para dim_territorio.",
        "sk_tempo": "FK para dim_tempo. Valor -1 indica contrato sem data de agendamento.",
        "sk_tipo_atividade": "FK para dim_tipo_atividade.",
        "sk_origem_sistema": "FK para dim_origem_sistema.",
        "documento": "CPF (11 dígitos) ou CNPJ (14 dígitos), tratado como texto. Dado sintético.",
        "tipo_documento": "PF ou PJ, derivado do tamanho do documento. Nulo para o Sistema D.",
        "status_unificado": "Domínio: agendado, atrasado, oportunidade. Unificado via de-para dos 4 sistemas.",
        "valor_recorrente": "Valor recorrente do contrato em R$. Nulo quando o sistema de origem não coleta.",
        "flag_replicado_multissistema": "True quando o contrato aparece em mais de um sistema de origem.",
    },
    "gold.dim_territorio": {
        "sk_territorio": "Chave surrogate, 1 a 5.",
        "territorio": "Território normalizado via de-para. Lista fechada de 5 valores.",
    },
    "gold.dim_tipo_atividade": {
        "sk_tipo_atividade": "Chave surrogate, 1 a 2.",
        "tipo_atividade": "Domínio: instalação, vistoria. Consolidado a partir de 6 rótulos originais.",
    },
    "gold.dim_origem_sistema": {
        "sk_origem_sistema": "Chave surrogate, 1 a 4.",
        "sistema_origem": "Domínio: A, B, C, D.",
    },
    "gold.dim_tempo": {
        "sk_tempo": "Chave surrogate. -1 = linha sentinela Não aplicável.",
        "data_agendamento": "Data do agendamento. Nula na linha sentinela.",
        "ano": "Ano da data de agendamento.",
        "mes": "Mês da data de agendamento, 1 a 12.",
        "dia_semana": "Nome do dia da semana, ou Não aplicável na linha sentinela.",
    },
}

for tabela, comentario in comentarios_tabelas.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")

for tabela, colunas in comentarios_colunas.items():
    for coluna, comentario in colunas.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{comentario}'")

print("Catálogo documentado: 10 tabelas e 22 colunas com descrição")